# Phòng lab: EDA & Data Mining hiệu quả đầu tư
**Sinh viên: NTTT~B23DCKD069**

Quy trình: **Khám phá → Làm sạch → Kết hợp → Tạo chỉ số → Khai phá insight → SQLite**.

Câu hỏi nghiên cứu: quỹ nào tăng trưởng tốt trong dữ liệu quan sát? Mức sụt giảm đi kèm là bao nhiêu? Lợi nhuận 12 tháng có vượt CPI YoY cùng kỳ không?

Notebook này dùng đúng các hàm sẽ chạy trong nút cập nhật Streamlit. Các nhận xét là kết quả phân tích bộ CSV được cung cấp, không xác nhận tính chính xác của nguồn ngoài.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display
from src.cleaner import read_raw, clean_data, parse_number
from src.transformer import transform_data, period_summary
from src.pipeline import run_pipeline
from src.database import load_database
pd.set_option('display.max_columns', 20)
raw = read_raw(ROOT / 'data/raw')

## 1. Khám phá dữ liệu và đơn vị quan sát
Trước khi tính toán, em xác định khóa: `(fund_id, ngày)` cho NAV; `(ngân hàng, hình thức, kỳ hạn, thời điểm quét)` cho lãi suất; tháng cho CPI. NAV có tần suất quan sát không đều nên không mặc định mọi bản ghi là một ngày giao dịch liên tiếp.

`raw_csv/` là bản ban đầu của đề bài; `data/raw/` là đầu vào vận hành. Năm file được đọc ở dạng chuỗi để nhìn thấy HTML, dấu phẩy thập phân và ô thiếu trước khi ép kiểu.

In [2]:
inventory = pd.DataFrame([
    {'Bảng': name, 'Dòng': len(df), 'Cột': len(df.columns),
     'Ô thiếu': int(df.isna().sum().sum()), 'Trùng hoàn toàn': int(df.duplicated().sum())}
    for name, df in raw.items()
])
display(inventory)
for name, frame in raw.items():
    print(name)
    display(frame.head(3))

,Bảng,Dòng,Cột,Ô thiếu,Trùng hoàn toàn
0,fund_info,5,14,0,0
1,fund_nav,4871,4,0,10
2,fund_holdings,14,4,0,0
3,bank_rates,45,14,0,0
4,macro_cpi,82,4,1,0


fund_info


,fund_id,symbol,fund_name,fund_type,risk_level,buy_min_value,management_fee,owner_name,return_1m,return_3m,return_6m,return_12m,return_since_inception,is_transferred
0,32,VCBF-BCF,QUỸ ĐẦU TƯ CỔ PHIẾU HÀNG ĐẦU VCBF,Quỹ cổ phiếu,Cao,100000.0,1.9,CÔNG TY TNHH QUẢN LÝ QUỸ ĐẦU TƯ CHỨNG KHOÁN VI...,-3.47,-6.92,-6.94,-5.8,305.35,True
1,21,VFF,QUỸ ĐẦU TƯ TRÁI PHIẾU NÂNG CAO VINACAPITAL,Quỹ trái phiếu,Thấp - Trung bình,100000.0,0.95,CÔNG TY CỔ PHẦN QUẢN LÝ QUỸ VINACAPITAL,1.02,2.36,4.25,8.17,170.35,True
2,23,VESAF,QUỸ ĐẦU TƯ CỔ PHIẾU TĂNG TRƯỞNG CHIẾN LƯỢC VIN...,Quỹ cổ phiếu,Trung bình-Cao,100000.0,1.75,CÔNG TY CỔ PHẦN QUẢN LÝ QUỸ VINACAPITAL,-1.44,-5.3,-9.99,-9.72,213.94,True


fund_nav


,fund_id,symbol,navDate,nav
0,32,VCBF-BCF,2020-01-02,18294.12
1,32,VCBF-BCF,2020-01-08,18353.8
2,32,VCBF-BCF,2020-01-15,18399.89


...

,fund_id,asset_type,asset_percent,update_date
0,32,Cổ phiếu,94.75,2026-09-09
1,32,Khác,0.46,2026-09-09
2,32,Tiền và tương đương tiền,4.79,2026-09-09


bank_rates


,deposit_type,bank_name,Kỳ hạn gửi tiết kiệm (tháng),Không Kỳ Hạn,01 tháng,03 tháng,06 tháng,09 tháng,12 tháng,13 tháng,18 tháng,24 tháng,web_update_time,scrape_date
0,OTC,ABBank,"0,10","3,60","3,80","6,05","6,05","6,05","6,05","5,60","5,60","5,60",2026-10-06 18:50:22,2026-10-06 21:46:55
1,OTC,Agribank,"0,20","2,60","2,90","4,00","4,00","5,90","5,90","5,90","6,00",-,2026-10-06 18:50:22,2026-10-06 21:46:55
2,OTC,Bắc Á,"<span class=""text-green"">0,50</span>","<span class=""text-green"">4,75</span>","<span class=""text-green"">4,75</span>","<span class=""text-green"">7,05</span>","<span class=""text-green"">7,05</span>","7,10","6,95","6,95","6,95","6,95",2026-10-06 18:50:22,2026-10-06 21:46:55


macro_cpi


,period_str,cpi_yoy_str,cpi_mom_str,status
0,Tháng 1/2020,"2,89%","0,93%",Chính thức
1,Tháng 2/2020,"4,29%",NaN,Chính thức
2,Tháng 3/2020,"2,74%","0,88%",Chính thức


## 2. Chẩn đoán chất lượng trước khi sửa
Em quan sát lãi suất có thẻ `<span>`, dấu phẩy thập phân và dấu `-`. Cột **Kỳ hạn gửi tiết kiệm (tháng)** lại chứa các số 0,10 / 0,20, trong khi cột Không Kỳ Hạn chứa mức vài phần trăm: đây là dấu hiệu tiêu đề lệch khi cào dữ liệu. Không có HTML nguồn hay sơ đồ ánh xạ đáng tin cậy nên em giữ nhãn kỳ hạn gốc, lưu cột không rõ kỳ hạn riêng, và cảnh báo mọi kết quả ngân hàng cần xác minh.

Giá NAV trùng khóa nhưng khác giá trị không được lấy trung bình hoặc chọn tùy ý: cả hai bản sẽ được cách ly. Dòng trùng hoàn toàn chỉ giữ một. CPI MoM thiếu được giữ NULL, không điền 0.

In [3]:
bank_raw = raw['bank_rates']
html_rows = bank_raw.apply(lambda col: col.str.contains('<[^>]+>', na=False)).any(axis=1)
display(bank_raw.loc[html_rows].head())
nav_raw = raw['fund_nav']
display(nav_raw[nav_raw.duplicated(['fund_id', 'navDate'], keep=False)].head(12))
display(raw['macro_cpi'][raw['macro_cpi'].cpi_mom_str.isna()])
examples = ['<span class="text-green">6,05</span>', '-0,21%', '-', '1.234,56']
display(pd.DataFrame({'Thô': examples, 'Sau parse_number': [parse_number(v) for v in examples]}))

,deposit_type,bank_name,Kỳ hạn gửi tiết kiệm (tháng),Không Kỳ Hạn,01 tháng,03 tháng,06 tháng,09 tháng,12 tháng,13 tháng,18 tháng,24 tháng,web_update_time,scrape_date
2,OTC,Bắc Á,"<span class=""text-green"">0,50</span>","<span class=""text-green"">4,75</span>","<span class=""text-green"">4,75</span>","<span class=""text-green"">7,05</span>","<span class=""text-green"">7,05</span>","7,10","6,95","6,95","6,95","6,95",2026-10-06 18:50:22,2026-10-06 21:46:55
5,OTC,GPBank,"<span class=""text-green"">0,50</span>","3,45","3,55","4,90","5,00","5,20","5,20","5,20","5,20","5,20",2026-10-06 18:50:22,2026-10-06 21:46:55
6,OTC,Hong Leong,-,"4,50","<span class=""text-green"">4,75</span>","6,30","6,30","6,50","6,00","6,00","6,00","6,00",2026-10-06 18:50:22,2026-10-06 21:46:55
8,OTC,Kiên Long,"<span class=""text-green"">0,50</span>","3,50","3,50","5,30","4,90","5,50","5,15","5,05","5,05","5,10",2026-10-06 18:50:22,2026-10-06 21:46:55
10,OTC,MB,"0,10","3,20","3,60","4,20","4,20","4,85","<span class=""text-red"">4,85</span>","4,85","5,70","5,70",2026-10-06 18:50:22,2026-10-06 21:46:55


,fund_id,symbol,navDate,nav
45,32,VCBF-BCF,2020-09-30,17682.3
46,32,VCBF-BCF,2020-09-30,17764.46
77,32,VCBF-BCF,2021-03-31,23512.01
78,32,VCBF-BCF,2021-03-31,23629.82
93,32,VCBF-BCF,2021-06-30,28351.1
94,32,VCBF-BCF,2021-06-30,28300.25
742,21,VFF,2020-11-03,17594.23
743,21,VFF,2020-11-03,17594.23
744,21,VFF,2020-11-10,17616.74
745,21,VFF,2020-11-10,17616.74


,period_str,cpi_yoy_str,cpi_mom_str,status
1,Tháng 2/2020,"4,29%",NaN,Chính thức


,Thô,Sau parse_number
0,"<span class=""text-green"">6,05</span>",6.05
1,"-0,21%",-0.21
2,-,NaN
3,"1.234,56",1234.56


## 3. Làm sạch và kết hợp (Stage 3)
Em tách việc đọc file khỏi biến đổi để dễ kiểm thử. Ngày được ép `datetime`, NAV và tỷ lệ được ép `float`; giá NAV không dương bị loại. Kết hợp NAV và metadata theo `fund_id` với kiểm tra `many_to_one` để tránh nhân bản số dòng. CPI gắn theo tháng để đối chiếu hồi cứu, không được hiểu là đã công bố tại ngày giao dịch.

Báo cáo dưới đây ghi số dòng loại/cách ly và thiếu dữ liệu, thay vì khẳng định dữ liệu đã hoàn hảo.

In [4]:
clean = clean_data(raw)
display(clean['quality_report'])
display(clean['rejected_rows'])
display(clean['fund_nav'].dtypes.to_frame('Kiểu dữ liệu'))
assert not clean['fund_nav'].duplicated(['fund_id', 'date']).any()
assert clean['fund_nav'].nav.gt(0).all()
print('Số NAV dùng được:', len(clean['fund_nav']))

,table_name,detail,count
0,fund_info,Dòng thiếu hoặc sai giá trị bắt buộc bị loại,0
1,fund_info,Dòng trùng hoàn toàn bị loại,0
2,fund_info,Dòng khóa trùng mâu thuẫn được cách ly,0
3,fund_nav,Dòng thiếu hoặc sai giá trị bắt buộc bị loại,0
4,fund_nav,Dòng trùng hoàn toàn bị loại,10
5,fund_nav,Dòng khóa trùng mâu thuẫn được cách ly,18
6,fund_holdings,Dòng thiếu hoặc sai giá trị bắt buộc bị loại,0
7,fund_holdings,Dòng trùng hoàn toàn bị loại,0
8,fund_holdings,Dòng khóa trùng mâu thuẫn được cách ly,0
9,fund_holdings,Cơ cấu có tổng khác 100% (sai số > 0.1),0


,table_name,reason,record_json
0,fund_nav,Khóa trùng mâu thuẫn,"{""fund_id"": 32.0, ""symbol"": ""VCBF-BCF"", ""date""..."
1,fund_nav,Khóa trùng mâu thuẫn,"{""fund_id"": 32.0, ""symbol"": ""VCBF-BCF"", ""date""..."
2,fund_nav,Khóa trùng mâu thuẫn,"{""fund_id"": 32.0, ""symbol"": ""VCBF-BCF"", ""date""..."
3,fund_nav,Khóa trùng mâu thuẫn,"{""fund_id"": 32.0, ""symbol"": ""VCBF-BCF"", ""date""..."
4,fund_nav,Khóa trùng mâu thuẫn,"{""fund_id"": 32.0, ""symbol"": ""VCBF-BCF"", ""date""..."
5,fund_nav,Khóa trùng mâu thuẫn,"{""fund_id"": 32.0, ""symbol"": ""VCBF-BCF"", ""date""..."
6,fund_nav,Khóa trùng mâu thuẫn,"{""fund_id"": 28.0, ""symbol"": ""DCDS"", ""date"": ""2..."
7,fund_nav,Khóa trùng mâu thuẫn,"{""fund_id"": 28.0, ""symbol"": ""DCDS"", ""date"": ""2..."
8,fund_nav,Khóa trùng mâu thuẫn,"{""fund_id"": 28.0, ""symbol"": ""DCDS"", ""date"": ""2..."
9,fund_nav,Khóa trùng mâu thuẫn,"{""fund_id"": 28.0, ""symbol"": ""DCDS"", ""date"": ""2..."


,Kiểu dữ liệu
fund_id,float64
date,datetime64[ns]
nav,float64
symbol,object
fund_type,object


Số NAV dùng được: 4843


## 4. Tạo chỉ số có cùng cơ sở so sánh (Stage 4)
**Normalized NAV = NAV(t) / NAV(gốc) × 100.** Các quỹ có NAV tuyệt đối khác nhau nên cần chung thang tăng trưởng. Ngày gốc là ngày quan sát đầu của từng quỹ trong giai đoạn chọn, không giả định có giá ngày 01/01/2020.

**CAGR = (NAV cuối / NAV đầu)^(365,25 / số ngày) − 1.** Drawdown đo mức giảm từ đỉnh đã quan sát. Lợi nhuận 12M lấy giá tại hoặc trước mốc 12 tháng, cho phép độ trễ tối đa 14 ngày; thiếu lịch sử thì để trống.

**Lợi nhuận thực xấp xỉ = Return 12M − CPI YoY cùng tháng.** Fisher chính xác hơn: `(1+r)/(1+i)−1`. Không trừ CPI YoY một năm khỏi lợi nhuận lũy kế nhiều năm.

In [5]:
tables = transform_data(clean)
nav = tables['fund_nav']
summary = period_summary(nav).sort_values('total_return', ascending=False)
display(summary.round(2))
assert np.allclose(nav.groupby('fund_id').first().normalized_nav, 100)
px.line(nav, x='date', y='normalized_nav', color='symbol',
        title='Tăng trưởng chuẩn hóa Base 100').show()

,symbol,start,end,total_return,cagr,max_drawdown,observations,return_1m_nav,return_3m_nav,return_6m_nav,return_12m_nav
3,VESAF,2020-01-07,2026-10-06,153.48,14.78,-39.53,1353,-1.44,-5.30,-9.99,-9.72
1,DCDS,2020-01-01,2026-10-06,129.01,13.04,-43.23,1674,-2.55,-7.29,-5.92,-11.23
2,VCBF-BCF,2020-01-02,2026-10-06,121.57,12.49,-32.42,694,-3.47,-6.92,-6.94,-5.80
4,VFF,2020-01-07,2026-10-06,60.92,7.31,-0.36,596,1.02,2.36,4.25,8.17
0,DCBF,2020-01-01,2026-10-06,57.50,6.95,-2.72,526,0.85,2.08,3.98,7.54


## 5. EDA trực quan: phân phối, tương quan và cơ cấu
Em dùng lợi nhuận **tháng** để giảm ảnh hưởng tần suất NAV không đều. Chỉ tính khi có NAV của hai tháng liền nhau; không forward-fill các tháng thiếu. Tương quan là mô tả đồng biến, không chứng minh quan hệ nhân quả. Khi dữ liệu có ít quỹ, xếp hạng và ma trận tương quan dễ giải thích hơn một mô hình học máy không có mục tiêu rõ ràng.

In [6]:
monthly_prices = nav.pivot(index='date', columns='symbol', values='nav').resample('ME').last()
monthly_returns = monthly_prices.pct_change(fill_method=None) * 100
display(monthly_returns.describe().round(2))
px.imshow(monthly_returns.corr(), text_auto='.2f', zmin=-1, zmax=1,
          color_continuous_scale='RdBu_r', title='Tương quan lợi nhuận tháng').show()
px.box(monthly_returns.melt(var_name='Quỹ', value_name='Lợi nhuận tháng (%)').dropna(),
       x='Quỹ', y='Lợi nhuận tháng (%)', title='Phân phối lợi nhuận tháng').show()
display(clean['fund_holdings'].pivot_table(index='symbol', columns='asset_type', values='asset_percent', aggfunc='last').fillna(0))

symbol,DCBF,DCDS,VCBF-BCF,VESAF,VFF
count,81.00,81.00,81.00,81.00,81.00
mean,0.56,1.25,1.22,1.34,0.59
std,0.28,6.71,6.13,6.40,0.18
min,-1.03,-25.23,-24.60,-23.49,-0.11
25%,0.50,-2.59,-1.97,-2.38,0.49
50%,0.58,1.40,2.00,1.36,0.56
75%,0.68,5.39,5.19,5.83,0.66
max,1.05,14.98,15.82,14.30,1.19


asset_type,Cổ phiếu,Khác,Tiền và tương đương tiền,Trái phiếu
symbol,,,,
DCBF,0.00,0.00,26.20,73.80
DCDS,90.07,0.12,8.97,0.84
VCBF-BCF,94.75,0.46,4.79,0.00
VESAF,88.91,0.27,10.82,0.00
VFF,0.00,0.00,23.58,76.42


## 6. Data Mining: xếp hạng và phát hiện đánh đổi tăng trưởng–sụt giảm
Em khai phá quy luật theo nhóm quỹ và dùng **biên Pareto**: một quỹ bị trội nếu có quỹ khác vừa CAGR cao hơn hoặc bằng, vừa sụt giảm tối đa ít nghiêm trọng hơn hoặc bằng, với ít nhất một tiêu chí tốt hơn. Đây là cách sàng lọc mô tả trên mẫu hiện có, không phải khuyến nghị mua quỹ.

Các ngày NAV mâu thuẫn đã cách ly có thể ảnh hưởng biến động. Kết quả drawdown chỉ đo trên các quan sát còn lại, không khẳng định đã ghi nhận mọi đáy thị trường.

In [7]:
mining = summary.merge(clean['fund_info'][['symbol', 'fund_type']], on='symbol')
mining['pareto'] = [not (((mining.cagr >= row.cagr) &
                          (mining.max_drawdown >= row.max_drawdown) &
                          ((mining.cagr > row.cagr) | (mining.max_drawdown > row.max_drawdown))).any())
                    for row in mining.itertuples()]
display(mining[['symbol', 'fund_type', 'total_return', 'cagr', 'max_drawdown', 'pareto']].round(2))
px.scatter(mining, x='max_drawdown', y='cagr', color='fund_type', text='symbol',
           symbol='pareto', title='Tăng trưởng và mức giảm từ đỉnh quan sát').show()
leader = mining.iloc[0]
print(f'Quỹ dẫn đầu tổng lợi nhuận trong dữ liệu: {leader.symbol}, {leader.total_return:.2f}%.')
print('Các quỹ trên biên Pareto:', ', '.join(mining.loc[mining.pareto, 'symbol']))

,symbol,fund_type,total_return,cagr,max_drawdown,pareto
0,VESAF,Quỹ cổ phiếu,153.48,14.78,-39.53,True
1,DCDS,Quỹ cổ phiếu,129.01,13.04,-43.23,False
2,VCBF-BCF,Quỹ cổ phiếu,121.57,12.49,-32.42,True
3,VFF,Quỹ trái phiếu,60.92,7.31,-0.36,True
4,DCBF,Quỹ trái phiếu,57.50,6.95,-2.72,False


Quỹ dẫn đầu tổng lợi nhuận trong dữ liệu: VESAF, 153.48%.
Các quỹ trên biên Pareto: VESAF, VCBF-BCF, VFF


## 7. Insight sức mua: cùng tháng, cùng kỳ 12M
Để tránh so sánh lệch thời điểm, em chọn tháng gần nhất có lợi nhuận 12M của tất cả quỹ, và luôn hiển thị trạng thái CPI sơ bộ/chính thức. Lợi nhuận thực âm nghĩa là mức tăng NAV trong kỳ không bù CPI YoY; không có nghĩa NAV nhất thiết giảm tuyệt đối.

In [8]:
monthly = nav.sort_values('date').groupby(['symbol', 'month']).tail(1)
valid = monthly.dropna(subset=['real_return_12m'])
counts = valid.groupby('month').symbol.nunique()
common = counts[counts == nav.symbol.nunique()]
if not common.empty:
    latest = valid[valid.month == common.index.max()]
    display(latest[['symbol', 'month', 'return_12m_nav', 'cpi_yoy', 'cpi_status', 'real_return_12m', 'real_return_fisher_12m']].round(2))
    px.bar(latest.sort_values('real_return_12m'), x='symbol', y='real_return_12m',
           color='fund_type', title='Lợi nhuận thực 12M xấp xỉ ở tháng chung gần nhất').show()
    print('Số quỹ vượt CPI:', int((latest.real_return_12m > 0).sum()), '/', len(latest))
else:
    print('Chưa có tháng chung đủ lịch sử cho tất cả quỹ.')

,symbol,month,return_12m_nav,cpi_yoy,cpi_status,real_return_12m,real_return_fisher_12m
4148,DCDS,2026-10-01,-11.23,2.81,Số liệu sơ bộ,-14.04,-13.65
1948,VESAF,2026-10-01,-9.72,2.81,Số liệu sơ bộ,-12.53,-12.19
2474,DCBF,2026-10-01,7.54,2.81,Số liệu sơ bộ,4.73,4.60
595,VFF,2026-10-01,8.17,2.81,Số liệu sơ bộ,5.36,5.21
4842,VCBF-BCF,2026-10-01,-5.80,2.81,Số liệu sơ bộ,-8.61,-8.37


Số quỹ vượt CPI: 2 / 5


## 8. Đóng gói vận hành và lưu SQLite
Notebook chứng minh phép biến đổi; `src/cleaner.py` và `src/transformer.py` là nơi giữ logic dùng chung. Pipeline ghi toàn bộ snapshot và lịch sử cập nhật trong một transaction. Nếu đọc nguồn, schema hoặc ghi dữ liệu lỗi, snapshot trước vẫn dùng được.

Để thí nghiệm không thay CSDL của ứng dụng, ô sau chạy pipeline vào CSDL tạm, rồi đọc lại để đối chiếu.

In [9]:
import tempfile
with tempfile.TemporaryDirectory() as temporary:
    db = Path(temporary) / 'lab.db'
    run = run_pipeline(db_path=db)
    loaded = load_database(db)
    assert len(loaded['fund_nav']) == len(nav)
    display(pd.DataFrame([{'Bảng SQLite': name, 'Dòng': len(frame)} for name, frame in loaded.items()]))
    print('Chế độ thu thập:', run['mode'])

,Bảng SQLite,Dòng
0,macro_cpi,82
1,fund_nav,4843
2,fund_holdings,14
3,fund_info,5
4,rejected_rows,18
5,bank_rates,405
6,quality_report,19
7,update_history,1


Chế độ thu thập: File CSV cục bộ


## 9. Kết luận và giới hạn
- Kết luận về quỹ dẫn đầu và sức mua được sinh từ dữ liệu ở các ô trên, không hard-code số ví dụ trong bản thiết kế.
- NAV có khóa trùng mâu thuẫn; các dòng đó được cách ly và giữ trong bảng kiểm toán.
- Snapshot lãi suất không đại diện lịch sử 2020–2026; tiêu đề kỳ hạn cần đối chiếu HTML nguồn. Chỉ sử dụng làm kịch bản giả định có ghi nhãn.
- CPI có số sơ bộ và thiếu MoM; chưa đủ căn cứ dựng sức mua tích lũy toàn kỳ từ chỉ số giá tháng.
- Phí quản lý và thông tin quỹ là snapshot không có ngày hiệu lực. Return trong factsheet có thể khác lợi nhuận tự tính vì thời điểm/chuỗi NAV khác nhau. Không trừ thêm phí quản lý lần nữa vào NAV khi chưa rõ quy ước nguồn.
- Bước tiếp theo khi có nguồn gốc rõ ràng: sửa ánh xạ kỳ hạn, bổ sung lãi suất theo thời gian và ngày công bố CPI để phân tích không nhìn trước tương lai.